In [1]:
import pandas as pd
import time
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

In [2]:
def get_darko_data():
    print("--- Starting Scraper ---")

    chrome_options = Options()
    # Remove the line below to have Selenium run through a new browser, otherwise can run in background
    chrome_options.add_argument("--headless")

    # Regular Chrome setup for no bot detection
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--window-size=1920,1080")

    # Launch browser and navigate
    # You may have to adjust the following line to fit your version of MacOS or Chrome or Safari
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=chrome_options
    )

    try:
        url = "https://www.nbarapm.com/datasets/MetricHistory"
        print(f"Navigating to {url}...")
        driver.get(url)

        wait = WebDriverWait(driver, 15)

        print("Looking for DARKO button...")

        wait.until(EC.presence_of_element_located((By.ID, "MetricHistory")))

        # Find and click DARKO tab (can change to MAMBA, RAPTOR or LEBRON)
        button_xpath = "//*[@id='MetricHistory']//*[contains(text(), 'DARKO')]"

        darko_btn = wait.until(EC.element_to_be_clickable((By.XPATH, button_xpath)))

        driver.execute_script("arguments[0].scrollIntoView(true);", darko_btn)
        time.sleep(
            1
        )  # Add line to give some time in between iterations/actions executed

        driver.execute_script("arguments[0].click();", darko_btn)
        print("Clicked DARKO tab.")

        target_id = "DarkoFull"

        print(f"Waiting for table #{target_id}...")
        wait.until(EC.presence_of_element_located((By.ID, target_id)))

        time.sleep(3)

        # Extract data via JavaScript
        print("Extracting data from Tabulator...")
        js_script = f"""
        // Try to find the table instance
        var table = Tabulator.findTable("#{target_id}")[0];
        
        if (table) {{
            return JSON.stringify(table.getData());
        }} else {{
            // Fallback: Check if there's a global variable or different selector
            return "NULL";
        }}
        """

        json_data = driver.execute_script(js_script)

        # Error handling
        if json_data == "NULL" or not json_data:
            print("Error: Tabulator instance not found.")
            print("Debug: Dumping page source to 'debug.html'...")
            with open("debug.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            return None

        data = json.loads(json_data)
        df = pd.DataFrame(data)

        print(f"✅ Success! Extracted {len(df)} rows.")
        return df

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

    # MUST: run this for the browser to close
    finally:
        driver.quit()

In [4]:
if __name__ == "__main__":
    df = get_darko_data()

    # Print to CSV
    if df is not None:
        print(df.head())
        df.to_csv("data/darko_history.csv", index=False)
        print("Saved to darko_history.csv")

--- Starting Scraper ---
Navigating to https://www.nbarapm.com/datasets/MetricHistory...
Looking for DARKO button...
Clicked DARKO tab.
Waiting for table #DarkoFull...
Extracting data from Tabulator...
✅ Success! Extracted 15054 rows.
    age  box_ddpm  box_odpm  d_dpm  d_dpm_rank   dpm  dpm_rank  nba_id  o_dpm  \
0  36.1     -0.34      1.47  -0.61         301  0.43        73       2   1.04   
1  31.1     -0.16     -0.81  -0.13         160 -0.91       170       3  -0.78   
2  38.0     -0.49     -2.36  -0.47         254 -2.90       405       7  -2.43   
3  35.7     -0.54     -1.44  -0.52         273 -2.01       299       9  -1.49   
4  27.3     -0.52     -1.22  -0.52         273 -1.72       261      12  -1.19   

   o_dpm_rank  on_off_ddpm  on_off_odpm     player_name  season  \
0          50        -1.14        -0.34     Byron Scott    1997   
1         199        -0.06        -0.65      Grant Long    1997   
2         389        -0.33        -3.06     Dan Schayes    1997   
3         